# Clean the HRRR-AK NetCDF: required checks and CRS metadata

**Purpose:** check the April 1–7, 2019 small-domain dataset, restore its native spatial coordinates and CRS metadata, and save a verified cleaned copy. Point selection and plotting are in **`4_explore_hrrrak_netcdf.ipynb`**.

| Input | Output |
| :--- | :--- |
| `hrrrak/lone_island_small_hrrrak_20190401_20190407.nc` | Same folder and filename with `_crs.nc` suffix |

“Cleaning” here does not change meteorological values, fill gaps, convert units, or reproject/resample the grid. The native grid is polar stereographic; projection information comes from an original GRIB file referenced by the NetCDF.

Select **glacierbay**, then **Run All** and save the notebook with its real outputs for GitHub. Local source files are required; no network access is used. The original NetCDF remains unchanged. Rerunning replaces the derived cleaned file.

## 1. Import tools and check the environment

In [1]:
from pathlib import Path
import sys
from datetime import datetime, timezone

if Path(sys.prefix).name != "glacierbay":
    raise RuntimeError("Select the glacierbay kernel, restart, and run all cells.")

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # Registers .rio for CRS metadata management.
from pyproj import CRS, Transformer
from eccodes import codes_grib_new_from_file, codes_get, codes_release

print(f"Python: {sys.executable}")

Python: /home/cassie/programs/miniforge3/envs/glacierbay/bin/python


## 2. Open the NetCDF

An xarray **Dataset** holds several variables. Each variable is a **DataArray**, with dimensions, coordinates, values, and attributes. `decode_coords="all"` also recognizes a CRS grid-mapping coordinate when one is present.

The file is small, so `.load()` reads it into memory before we close the file handle. Explore the summary below: which dimensions describe space, and which describes time?

In [2]:
# Resolve paths from the project folder or its parent workspace.
project_dir = Path.cwd()
if not (project_dir / "hrrrak").is_dir():
    project_dir = project_dir / "glacierbay_alaska"
data_dir = project_dir / "hrrrak"
input_path = data_dir / "lone_island_small_hrrrak_20190401_20190407.nc"

with xr.open_dataset(input_path, decode_coords="all", decode_timedelta=True) as opened:
    ds = opened.load()
# Use the project label for the Island reference point in displayed metadata.
ds.attrs["coordinate_provenance"] = "Island reference point; exact station location unconfirmed"
print(f"Opened: {input_path.resolve()}")
display(ds)

Opened: /home/cassie/python/repos/glacierbay_alaska/hrrrak/lone_island_small_hrrrak_20190401_20190407.nc


<xarray.Dataset> Size: 117kB
Dimensions:                  (time: 168, y: 3, x: 3)
Coordinates:
  * time                     (time) datetime64[ns] 1kB 2019-04-01T05:00:00 .....
    forecast_reference_time  (time) datetime64[ns] 1kB 2019-04-01 ... 2019-04...
    forecast_period          (time) timedelta64[ns] 1kB 05:00:00 ... 07:00:00
    source_file              (time) <U42 28kB 'hrrr.20190401/hrrr.t00z.wrfsfc...
    latitude                 (y, x) float64 72B 58.69 58.69 ... 58.74 58.75
    longitude                (y, x) float64 72B -136.3 -136.3 ... -136.3 -136.2
Dimensions without coordinates: y, x
Data variables: (12/14)
    temp                     (time, y, x) float32 6kB 283.6 283.8 ... 280.1
    rh                       (time, y, x) float32 6kB 50.7 50.2 ... 90.9 89.4
    u10                      (time, y, x) float32 6kB 3.406 4.594 ... -4.852
    v10                      (time, y, x) float32 6kB 2.294 0.6693 ... 4.817
    pressure                 (time, y, x) float32 6kB 1.031e+05 ... 9.982e+04
    temp_surface             (time, y, x) float32 6kB 281.3 281.5 ... 282.6
    ...                       ...
    lwrad                    (time, y, x) float32 6kB 262.1 262.5 ... 332.7
    precip_rate              (time, y, x) float32 6kB 0.0 0.0 ... 0.00019
    precip_total             (time, y, x) float32 6kB 0.0 0.0 ... 3.795 3.283
    snowdepth                (time, y, x) float32 6kB 0.0 0.0 0.0 ... 0.0 0.0
    swe                      (time, y, x) float32 6kB 0.0 0.0 0.0 ... 0.0 0.0
    wind                     (time, y, x) float32 6kB 4.107 4.642 ... 6.837
Attributes: (12/32)
    GRIB_paramId:                    167
    GRIB_dataType:                   fc
    GRIB_numberOfPoints:             9
    GRIB_typeOfLevel:                heightAboveGround
    GRIB_stepUnits:                  1
    GRIB_stepType:                   instant
    ...                              ...
    date_selection:                  Explicit dates
    missing_expected_files:          0
    partial_download:                0
    missing_file_list:               
    history:                         Created 2026-10-15T11:30:53.246850+00:00...
    processing_notes:                Native units retained. Precipitation tot...

## 3. Check values and forecast timing

A dataset that opens successfully still needs inspection. Look for missing values, unexpected ranges, duplicate forecast records, and time gaps. These checks report what exists; they do not fill gaps or remove unusual weather.

| Coordinate | Meaning |
| :--- | :--- |
| `forecast_reference_time` | Forecast initialization, in UTC |
| `forecast_period` | Lead time after initialization |
| `time` | UTC valid time: initialization + lead |

Time values are stored without a pandas time-zone suffix, but their documented convention is UTC. Do not treat them as Alaska local time. Duplicate **valid times** can represent different forecasts; duplicate initialization/lead pairs would be a problem.

In [3]:
# Basic structural checks before saving; report missing values without filling them.
if not {"time", "y", "x"}.issubset(ds.dims):
    raise ValueError("Expected time, y, x dimensions.")
if not all("units" in value.attrs for value in ds.data_vars.values()):
    raise ValueError("A meteorological variable is missing its units metadata.")
if not np.isfinite(ds.latitude.values).all() or not np.isfinite(ds.longitude.values).all():
    raise ValueError("Grid locations contain invalid coordinates.")

# Report ranges in native units and count missing values.
display(pd.DataFrame([
    {"variable": name, "units": value.attrs.get("units", ""),
     "minimum": float(value.min()), "maximum": float(value.max()),
     "missing": int(value.isnull().sum())}
    for name, value in ds.data_vars.items()
]))
timing = pd.DataFrame({
    "initialization_UTC": ds.forecast_reference_time.values,
    "lead_hours": ds.forecast_period.values / np.timedelta64(1, "h"),
    "valid_time_UTC": ds.time.values,
})
assert not timing.duplicated(["initialization_UTC", "lead_hours"]).any()
assert np.array_equal(ds.time.values, ds.forecast_reference_time.values + ds.forecast_period.values)
assert pd.Index(ds.time.values).is_monotonic_increasing
print("Spatial shape:", {key: ds.sizes[key] for key in ("y", "x")})
print("Duplicate valid times:", timing.valid_time_UTC.duplicated().sum())
print("Time-step counts (inspect gaps):")
print(timing.valid_time_UTC.diff().value_counts())
display(timing.head(10))

,variable,units,minimum,maximum,missing
0,temp,K,275.552246,285.607239,0
1,rh,%,40.299999,99.900002,0
2,u10,m s**-1,-9.363325,7.411646,0
3,v10,m s**-1,-5.284840,11.473278,0
4,pressure,Pa,98840.000000,103060.000000,0
5,temp_surface,K,281.058533,282.654449,0
6,terrain_height,m,0.000000,6.218750,0
7,swrad,W m**-2,0.000000,606.000000,0
8,lwrad,W m**-2,246.000000,337.899994,0
9,precip_rate,kg m**-2 s**-1,0.000000,0.001320,0


Spatial shape: {'y': 3, 'x': 3}
Duplicate valid times: 0
Time-step counts (inspect gaps):
valid_time_UTC
0 days 01:00:00    167
Name: count, dtype: int64


,initialization_UTC,lead_hours,valid_time_UTC
0,2019-04-01 00:00:00,5.0,2019-04-01 05:00:00
1,2019-04-01 00:00:00,6.0,2019-04-01 06:00:00
2,2019-04-01 00:00:00,7.0,2019-04-01 07:00:00
3,2019-04-01 03:00:00,5.0,2019-04-01 08:00:00
4,2019-04-01 03:00:00,6.0,2019-04-01 09:00:00
5,2019-04-01 03:00:00,7.0,2019-04-01 10:00:00
6,2019-04-01 06:00:00,5.0,2019-04-01 11:00:00
7,2019-04-01 06:00:00,6.0,2019-04-01 12:00:00
8,2019-04-01 06:00:00,7.0,2019-04-01 13:00:00
9,2019-04-01 09:00:00,5.0,2019-04-01 14:00:00


## 4. Investigate the CRS before setting it

A **CRS** tells us how coordinates locate places on Earth. A grid can have both projected coordinates in meters and auxiliary latitude/longitude in degrees.

Our processed file retains `GRIB_gridType="polar_stereographic"`, but initially has no formal CRS mapping or projected `x`/`y` coordinate values. The `y` and `x` dimension positions are array indices, not UTM eastings/northings. Having latitude/longitude arrays does **not** make this a regular geographic raster.

> **Assigning a CRS describes existing coordinates. Reprojecting changes the grid.** We will recover the native projection and coordinate values; we will not resample the weather fields or label the old indices as UTM.

The source GRIB records the projection, spherical Earth radius, and grid spacing. We inspect it rather than guessing an EPSG code. The code supports this sample's northern polar stereographic grid with spherical Earth shape 6; other grids need their own interpretation.

In [4]:
print("Embedded CRS before cleanup:", ds.rio.crs)
print("GRIB grid type:", ds.temp.attrs.get("GRIB_gridType"))
print("Existing x/y coordinates:", [name for name in ("x", "y") if name in ds.coords])

# Use the provenance coordinate to locate the first forecast's original GRIB.
source_grib = data_dir / str(ds.source_file.values[0])
keys = ["gridType", "shapeOfTheEarth", "radius", "LaDInDegrees",
        "orientationOfTheGridInDegrees", "projectionCentreFlag",
        "DxInMetres", "DyInMetres", "Nx", "Ny",
        "latitudeOfFirstGridPointInDegrees", "longitudeOfFirstGridPointInDegrees"]
with source_grib.open("rb") as handle:
    message = codes_grib_new_from_file(handle)
    try:
        grid = {key: codes_get(message, key) for key in keys}
    finally:
        codes_release(message)
display(pd.Series(grid).to_frame("GRIB grid metadata"))
if grid["gridType"] != "polar_stereographic" or grid["shapeOfTheEarth"] != 6 or grid["projectionCentreFlag"] != 0:
    raise ValueError("This walkthrough expects the sample's northern spherical polar stereographic grid.")
assert (grid["Ny"], grid["Nx"]) == ds.latitude.shape

Embedded CRS before cleanup: None
GRIB grid type: polar_stereographic
Existing x/y coordinates: []


,GRIB grid metadata
gridType,polar_stereographic
shapeOfTheEarth,6
radius,6371229
LaDInDegrees,60.0
orientationOfTheGridInDegrees,225.0
projectionCentreFlag,0
DxInMetres,3000.0
DyInMetres,3000.0
Nx,3
Ny,3


## 5. Restore native projected coordinates and write CRS metadata

We construct a polar stereographic CRS from the source metadata, using its **model sphere**, not assuming WGS84 or UTM. We transform the existing 2-D model longitude/latitude to projected coordinates, check that they form a regular grid, and assign the resulting 1-D `x` and `y` axes in meters.

A small tolerance allows rounding of GRIB geographic coordinates. We check the source's first grid point, grid spacing, and all reconstructed locations before writing the CRS.

We also remove dataset-level `GRIB_*` attributes inherited from the first temperature field: these describe a variable, not the whole dataset. Each variable retains its own GRIB metadata.

References: [pyproj coordinate transformation](https://pyproj4.github.io/pyproj/stable/api/transformer.html) and [rioxarray CRS management](https://corteva.github.io/rioxarray/stable/getting_started/crs_management.html).

In [5]:
# Recover the native CRS from evidence in the original GRIB header.
central_lon = (grid["orientationOfTheGridInDegrees"] + 180) % 360 - 180
native_crs = CRS.from_proj4(
    f"+proj=stere +lat_0=90 +lat_ts={grid['LaDInDegrees']} "
    f"+lon_0={central_lon} +R={grid['radius']} +units=m +no_defs"
)
# always_xy=True means longitude first, latitude second.
to_native = Transformer.from_crs(native_crs.geodetic_crs, native_crs, always_xy=True)
xx, yy = to_native.transform(ds.longitude.values, ds.latitude.values)
x_values, y_values = xx[0, :], yy[:, 0]
np.testing.assert_allclose(xx, np.broadcast_to(x_values, xx.shape), atol=1.0, rtol=0)
np.testing.assert_allclose(yy, np.broadcast_to(y_values[:, None], yy.shape), atol=1.0, rtol=0)
np.testing.assert_allclose(abs(np.diff(x_values)), grid["DxInMetres"], atol=1.0, rtol=0)
np.testing.assert_allclose(abs(np.diff(y_values)), grid["DyInMetres"], atol=1.0, rtol=0)
first_lon = (grid["longitudeOfFirstGridPointInDegrees"] + 180) % 360 - 180
np.testing.assert_allclose([ds.longitude.values[0, 0], ds.latitude.values[0, 0]],
                           [first_lon, grid["latitudeOfFirstGridPointInDegrees"]], atol=1e-5, rtol=0)

# Assign real projected coordinates and the matching CRS, without resampling data.
clean = ds.copy().assign_coords(x=x_values, y=y_values)
clean.attrs = {key: value for key, value in ds.attrs.items() if not key.startswith("GRIB_")}
clean = clean.rio.set_spatial_dims(x_dim="x", y_dim="y").rio.write_crs(native_crs)
clean = clean.rio.write_coordinate_system().rio.write_transform()
clean.attrs["crs_provenance"] = f"Native projection restored from {source_grib.name}; model sphere radius {grid['radius']} m"
clean.attrs["history"] = ds.attrs.get("history", "") + f"\n{datetime.now(timezone.utc).isoformat()}: restored native CRS and projected coordinates; no resampling."

# Inverse projection confirms that restored axes reproduce the supplied locations.
back = Transformer.from_crs(native_crs, native_crs.geodetic_crs, always_xy=True)
restored_lon, restored_lat = back.transform(*np.meshgrid(clean.x.values, clean.y.values))
np.testing.assert_allclose(restored_lon, clean.longitude.values, atol=2e-5, rtol=0)
np.testing.assert_allclose(restored_lat, clean.latitude.values, atol=2e-5, rtol=0)
np.testing.assert_array_equal(clean.temp.values, ds.temp.values)
print("Native CRS:", clean.rio.crs)
print("Grid spacing (meters):", clean.rio.resolution())
print("Temperature values unchanged:", np.array_equal(clean.temp.values, ds.temp.values))

Native CRS: PROJCS["unknown",GEOGCS["unknown",DATUM["unknown",SPHEROID["unknown",6371229,0]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433]],PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",60],PARAMETER["central_meridian",-135],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",SOUTH],AXIS["Northing",SOUTH]]
Grid spacing (meters): (3000.000000000269, 3000.0000000006985)
Temperature values unchanged: True


## 6. Save and verify the cleaned grid

Write a new NetCDF beside the input. A temporary file is checked before replacing the cleaned output. We verify the CRS, coordinates, times, and **all meteorological values and units** after reopening it.

The saved grid still contains every model cell and forecast. This notebook does not extract or save a point time series. Open **`4_explore_hrrrak_netcdf.ipynb`** next to inspect variables and attributes, select a cell near Lone Island, and make plots.

In [6]:
cleaned_path = data_dir / f"{input_path.stem}_crs.nc"
temporary = cleaned_path.with_suffix(".nc.tmp")
try:
    clean.to_netcdf(temporary, engine="netcdf4")
    with xr.open_dataset(temporary, decode_coords="all", decode_timedelta=True) as check:
        assert CRS.from_user_input(check.rio.crs).equals(native_crs)
        for name in ds.data_vars:
            np.testing.assert_array_equal(check[name].values, ds[name].values)
            assert check[name].attrs["units"] == ds[name].attrs["units"]
        for name in ("x", "y", "latitude", "longitude", "time", "forecast_reference_time", "forecast_period"):
            np.testing.assert_array_equal(check[name].values, clean[name].values)
    temporary.replace(cleaned_path)
finally:
    temporary.unlink(missing_ok=True)
print(f"Saved and verified: {cleaned_path.resolve()}")
print(f"Forecast records: {clean.sizes['time']}; spatial grid: {clean.sizes['y']} × {clean.sizes['x']}")

Saved and verified: /home/cassie/python/repos/glacierbay_alaska/hrrrak/lone_island_small_hrrrak_20190401_20190407_crs.nc
Forecast records: 168; spatial grid: 3 × 3
